In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler

In [0]:
print('=================LEITURA INICIAL DOS DADOS=================')

# Leer diretamente de Unity Catalog e não de um arquivo .csv
# Para importar o arquivo diretamente do repositório visite e faça download: https://archive.ics.uci.edu/dataset/186/wine+quality
df = spark.table('dbacademy.default.winequality_red').toPandas()

print('✅ Dados importados com sucesso desde Unity Catalog!')
print(f'📊 O dataset contém {df.shape[0]} linhas e {df.shape[1]} columnas')
print(f'📋 Columnas do dataset: {list(df.columns)}')

In [0]:
# Mostrando as principais colunas do dataset
print("\nInfo colunas do dataset:")
display(df.info()) 

# Estatisticas do dataset
print("Resumo estatístico:")
df.describe()

In [0]:
print('\n' + '='*70)
print('🧹 ETAPA 1: LIMPEZA INICIAL - REMOÇÃO DE COLUNA IRRELEVANTE')
print('='*70)

# Retirar a coluna wine_id que é irrelevante para o modelo
df_limpo = df.drop(columns=['wine_id'])

print(f'\n✅ Coluna wine_id removida')
print(f'📊 Dimensões atuais: {df_limpo.shape[0]} linhas × {df_limpo.shape[1]} colunas')
print(f'📋 Features restantes: {list(df_limpo.columns)}')

In [0]:
print('\n' + '='*70)
print('🔍 ETAPA 2: VERIFICAÇÃO DE VALORES NULOS E DUPLICATAS')
print('='*70)

# Verificar valores nulos
nulos = df_limpo.isnull().sum()
print(f'\n📌 Valores nulos por coluna:')
if nulos.sum() == 0:
    print('   ✅ Nenhum valor nulo encontrado!')
else:
    print(nulos[nulos > 0])
    print(f'\n⚠️ Total de células nulas: {nulos.sum()}')

# Verificar duplicatas
duplicatas = df_limpo.duplicated().sum()
print(f'\n📌 Linhas duplicadas: {duplicatas}')
if duplicatas > 0:
    print(f'⚠️ Removendo {duplicatas} linhas duplicadas...')
    df_limpo = df_limpo.drop_duplicates()
    print(f'✅ Duplicatas removidas. Shape atual: {df_limpo.shape}')
else:
    print('✅ Nenhuma duplicata encontrada!')

In [0]:


print('\n' + '='*70)
print('🚨 ETAPA 3: DETECÇÃO DE OUTLIERS (Z-SCORE > 3)')
print('='*70)

# Separar features numéricas (excluindo target)
features_numericas = df_limpo.drop(columns=['quality'])

# Calcular Z-Score
z_scores = np.abs(stats.zscore(features_numericas))

# Contar outliers por coluna
outliers_por_coluna = pd.DataFrame(z_scores > 3, columns=features_numericas.columns).sum()
total_outliers_linhas = (z_scores > 3).any(axis=1).sum()

print(f'\n📊 Outliers detectados por feature (Z > 3):')
for col in outliers_por_coluna[outliers_por_coluna > 0].index:
    print(f'   • {col}: {outliers_por_coluna[col]} outliers')

print(f'\n📈 Total de linhas com pelo menos 1 outlier: {total_outliers_linhas} ({total_outliers_linhas/len(df_limpo)*100:.1f}%)')
print(f'\n⚠️ Decisão: Remover outliers para melhorar qualidade do modelo')

# Remover outliers
df_sem_outliers = df_limpo[(z_scores < 3).all(axis=1)]

print(f'\n✅ Outliers removidos!')
print(f'   • Antes: {df_limpo.shape[0]} linhas')
print(f'   • Depois: {df_sem_outliers.shape[0]} linhas')
print(f'   • Removidas: {df_limpo.shape[0] - df_sem_outliers.shape[0]} linhas ({(df_limpo.shape[0] - df_sem_outliers.shape[0])/df_limpo.shape[0]*100:.1f}%)')

# Atualizar dataset
df_limpo = df_sem_outliers.copy()

In [0]:
print('\n' + '='*70)
print('⚖️ ETAPA 4: ANÁLISE DE BALANCEAMENTO DA VARIÁVEL TARGET')
print('='*70)

# Distribuição das classes
distribuicao = df_limpo['quality'].value_counts().sort_index()

print(f'\n📊 Distribuição da variável quality (target):')
for classe, count in distribuicao.items():
    percentual = (count / len(df_limpo)) * 100
    print(f'   Quality {classe}: {count:4d} vinhos ({percentual:5.1f}%)')

print(f'\n📌 Estatísticas de desbalanceamento:')
print(f'   • Classe mais frequente: {distribuicao.idxmax()} ({distribuicao.max()} ocorrências)')
print(f'   • Classe menos frequente: {distribuicao.idxmin()} ({distribuicao.min()} ocorrências)')
print(f'   • Proporção (max/min): {distribuicao.max() / distribuicao.min():.1f}:1')

if distribuicao.max() / distribuicao.min() > 10:
    print(f'\n⚠️ ALERTA: Dataset muito desbalanceado!')
    print(f'   Solução: Aplicaremos SMOTE (Synthetic Minority Over-sampling) após split')

# Visualização
plt.figure(figsize=(10, 6))
sns.countplot(x='quality', data=df_limpo, palette='viridis')
plt.title('Distribuição da Qualidade dos Vinhos (Target)', fontsize=14, fontweight='bold')
plt.xlabel('Quality (Qualidade)', fontsize=12)
plt.ylabel('Quantidade de Vinhos', fontsize=12)
plt.grid(axis='y', alpha=0.3)
for i, (classe, count) in enumerate(distribuicao.items()):
    plt.text(i, count + 10, str(count), ha='center', fontweight='bold')
plt.show()

In [0]:
print('\n' + '='*70)
print('🔗 ETAPA 5: ANÁLISE DE CORRELAÇÕES')
print('='*70)

# Correlação de Pearson (linear)
pearson_corr = df_limpo.corr(method='pearson')

# Correlação de Spearman (monotônica)
spearman_corr = df_limpo.corr(method='spearman')

# Correlações com target
print(f'\n📊 Top 5 features mais correlacionadas com QUALITY (Pearson):')
top_pearson = pearson_corr['quality'].drop('quality').sort_values(ascending=False).head(5)
for feat, corr in top_pearson.items():
    print(f'   • {feat:<25}: {corr:+.3f}')

print(f'\n📊 Top 5 features mais correlacionadas com QUALITY (Spearman):')
top_spearman = spearman_corr['quality'].drop('quality').sort_values(ascending=False).head(5)
for feat, corr in top_spearman.items():
    print(f'   • {feat:<25}: {corr:+.3f}')

# Heatmap comparativo
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pearson
sns.heatmap(pearson_corr, annot=True, fmt='.2f', cmap='Blues', 
            ax=axes[0], cbar_kws={'label': 'Correlação'}, linewidths=0.5)
axes[0].set_title('Correlação de Pearson (Linear)', fontsize=14, fontweight='bold', pad=15)

# Spearman
sns.heatmap(spearman_corr, annot=True, fmt='.2f', cmap='Oranges', 
            ax=axes[1], cbar_kws={'label': 'Correlação'}, linewidths=0.5)
axes[1].set_title('Correlação de Spearman (Monotônica)', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print(f'\n💡 INSIGHTS:')
print(f'   • alcohol tem a maior correlação positiva com quality')
print(f'   • volatile_acidity tem forte correlação negativa (vinhos ruins)')

In [0]:
print('\n' + '='*70)
print('✂️ ETAPA 6: SEPARAÇÃO DE FEATURES (X) E TARGET (y)')
print('='*70)

# Separar X (features) e y (target)
X = df_limpo.drop(columns=['quality'])
y = df_limpo['quality']

print(f'\n📊 Dimensões:')
print(f'   • Features (X): {X.shape}')
print(f'   • Target (y): {y.shape}')
print(f'\n📋 Features incluídas:')
for i, col in enumerate(X.columns, 1):
    print(f'   {i:2d}. {col}')

print(f'\n✅ Separação concluída!')

In [0]:
from sklearn.model_selection import train_test_split

print('\n' + '='*70)
print('🔀 ETAPA 7: DIVISÃO TREINO/TESTE (ESTRATIFICADA)')
print('='*70)

# Split estratificado (mantém proporção de classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 80% treino, 20% teste
    random_state=42,     # Reprodutibilidade
    stratify=y           # ⚠️ IMPORTANTE: mantém proporção de classes
)

print(f'\n📊 Divisão realizada:')
print(f'   • Total de amostras: {len(X):,}')
print(f'   • Treino: {len(X_train):,} amostras ({len(X_train)/len(X)*100:.1f}%)')
print(f'   • Teste: {len(X_test):,} amostras ({len(X_test)/len(X)*100:.1f}%)')

print(f'\n⚖️ Verificação do balanceamento (stratify):')
print(f'\n   Distribuição TREINO:')
for classe, count in y_train.value_counts().sort_index().items():
    print(f'      Quality {classe}: {count:3d} ({count/len(y_train)*100:5.1f}%)')

print(f'\n   Distribuição TESTE:')
for classe, count in y_test.value_counts().sort_index().items():
    print(f'      Quality {classe}: {count:3d} ({count/len(y_test)*100:5.1f}%)')

print(f'\n✅ Split estratificado concluído com sucesso!')

In [0]:
# Instalar imbalanced-learn se não estiver disponível
%pip install imbalanced-learn --quiet

from imblearn.over_sampling import SMOTE

print('\n' + '='*70)
print('🔄 ETAPA 8: BALANCEAMENTO COM SMOTE (APENAS NO TREINO)')
print('='*70)

print(f'\n📊 Distribuição ANTES do SMOTE (Treino):')
for classe, count in y_train.value_counts().sort_index().items():
    print(f'   Quality {classe}: {count:4d} vinhos')

# Aplicar SMOTE APENAS no conjunto de treino
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f'\n📊 Distribuição DEPOIS do SMOTE (Treino):')
for classe, count in pd.Series(y_train_balanced).value_counts().sort_index().items():
    print(f'   Quality {classe}: {count:4d} vinhos')

print(f'\n📈 Estatísticas:')
print(f'   • Amostras ANTES: {len(X_train):,}')
print(f'   • Amostras DEPOIS: {len(X_train_balanced):,}')
print(f'   • Amostras geradas: {len(X_train_balanced) - len(X_train):,} ({(len(X_train_balanced) - len(X_train))/len(X_train)*100:.1f}%)')

print(f'\n⚠️ IMPORTANTE: Teste permanece inalterado ({len(X_test)} amostras)')
print(f'✅ SMOTE aplicado com sucesso!')

In [0]:


print('\n' + '='*70)
print('📏 ETAPA 9: NORMALIZAÇÃO COM STANDARDSCALER')
print('='*70)

print(f'\n📊 Estatísticas ANTES da normalização (amostra do treino):')
print(X_train_balanced[:5].describe().loc[['mean', 'std']].T.head(5))

# Criar StandardScaler
scaler = StandardScaler()

# Fit no TREINO, transform em AMBOS
X_train_scaled = scaler.fit_transform(X_train_balanced)  # Fit + Transform
X_test_scaled = scaler.transform(X_test)                 # Apenas Transform

# Converter de volta para DataFrame (mantém nomes das colunas)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print(f'\n📊 Estatísticas DEPOIS da normalização (treino escalado):')
print(f'   • Média global: {X_train_scaled.mean().mean():.6f} (próximo de 0 ✅)')
print(f'   • Desvio padrão global: {X_train_scaled.std().mean():.6f} (próximo de 1 ✅)')

print(f'\n📋 Verificação por feature (primeiras 5):')
for col in X_train_scaled.columns[:5]:
    print(f'   • {col:<25}: média={X_train_scaled[col].mean():.4f}, std={X_train_scaled[col].std():.4f}')

print(f'\n✅ Normalização concluída!')
print(f'   • Treino normalizado: {X_train_scaled.shape}')
print(f'   • Teste normalizado: {X_test_scaled.shape}')
print(f'\n⚠️ IMPORTANTE: Scaler foi fit APENAS no treino (evita data leakage)')

In [0]:
print('\n' + '='*70)
print('🎯 RESUMO FINAL: DADOS PRONTOS PARA MACHINE LEARNING')
print('='*70)

print(f'\n📊 DIMENSÕES FINAIS:')
print(f'   • X_train (normalizado + balanceado): {X_train_scaled.shape}')
print(f'   • y_train (balanceado): {y_train_balanced.shape}')
print(f'   • X_test (normalizado): {X_test_scaled.shape}')
print(f'   • y_test (original): {y_test.shape}')

print(f'\n🔧 TRATAMENTOS APLICADOS:')
print(f'   ✅ 1. Remoção de wine_id (coluna irrelevante)')
print(f'   ✅ 2. Verificação de nulos e duplicatas')
print(f'   ✅ 3. Remoção de outliers (Z-Score > 3)')
print(f'   ✅ 4. Análise de balanceamento de classes')
print(f'   ✅ 5. Análise de correlações (Pearson + Spearman)')
print(f'   ✅ 6. Separação X/y')
print(f'   ✅ 7. Train/Test Split estratificado (80/20)')
print(f'   ✅ 8. SMOTE (balanceamento apenas no treino)')
print(f'   ✅ 9. StandardScaler (normalização)')

print(f'\n💡 PRÓXIMOS PASSOS:')
print(f'   📌 Treinar modelos de classificação:')
print(f'      • Logistic Regression')
print(f'      • Random Forest Classifier')
print(f'      • Decision Tree Classifier')
print(f'   📌 Avaliar com:')
print(f'      • Accuracy')
print(f'      • Classification Report (precision, recall, f1-score)')
print(f'      • Confusion Matrix')

print(f'\n🚀 Dataset pronto para treinamento!')
print('='*70)